# PPO: Proximal Policy Optimization

**Paper**: Schulman et al., 2017 — *Proximal Policy Optimization Algorithms* (OpenAI)

## Why Not Just Use Policy Gradient?

Vanilla REINFORCE updates the policy with:
$$\nabla J = E[\nabla \log \pi(a|s) \cdot A(s,a)]$$

A large gradient step can **destroy** the policy — it enters a bad region and never recovers. Using a tiny learning rate helps but makes training very slow.

## PPO: Clip the Update

Define the probability ratio between new and old policy:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

Clip it so the policy cannot change too much in one step:
$$L^{CLIP} = E\left[\min\left(r_t \hat{A}_t,\; \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon)\hat{A}_t\right)\right]$$

- If $r_t > 1+\varepsilon$: already moved toward a better action too much — stop pushing
- If $r_t < 1-\varepsilon$: already moved away from a bad action too much — stop pushing

## Actor-Critic Architecture

- **Actor** $\pi_\theta(a|s)$: outputs action logits (policy)
- **Critic** $V_\phi(s)$: estimates state value (baseline for variance reduction)
- **Advantage** $A_t = R_t - V(s_t)$: how much better than average was this action?

Total loss:
$$L = -L^{CLIP} + c_1 L^{value} - c_2 H[\pi_\theta]$$

Entropy bonus $H$ encourages exploration throughout training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

torch.manual_seed(42)
np.random.seed(42)

## Actor-Critic Network

Shared backbone (lower layers) with two heads:
- **Actor head**: outputs logits for each action
- **Critic head**: outputs scalar state value V(s)

Sharing lower layers is more parameter-efficient and often improves performance.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),    nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, action_dim)
        self.critic = nn.Linear(hidden, 1)

        # Orthogonal init — standard for PPO stability
        for layer in self.backbone:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.constant_(layer.bias, 0)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        feat   = self.backbone(x)
        logits = self.actor(feat)
        value  = self.critic(feat).squeeze(-1)
        return logits, value

    def get_action(self, state):
        logits, value = self(state)
        dist   = Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value

    def evaluate(self, states, actions):
        logits, values = self(states)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions), values, dist.entropy()

## Rollout Buffer + GAE

PPO is **on-policy**: collect N timesteps, update, discard, repeat. No replay buffer.

**GAE (Generalized Advantage Estimation)**:
$$A_t = \sum_{l=0}^{T} (\gamma \lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

- lambda=1: Monte Carlo (high variance, low bias)
- lambda=0: TD(0) (low variance, high bias)
- lambda=0.95: good practical balance

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states    = []
        self.actions   = []
        self.log_probs = []
        self.rewards   = []
        self.values    = []
        self.dones     = []

    def push(self, state, action, log_prob, reward, value, done):
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_gae(self, last_value, gamma=0.99, lam=0.95):
        advantages = []
        gae = 0.0
        values = self.values + [last_value]
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t+1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * lam * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def get_tensors(self, advantages, returns):
        states    = torch.FloatTensor(np.array(self.states)).to(device)
        actions   = torch.LongTensor(self.actions).to(device)
        log_probs = torch.FloatTensor(self.log_probs).to(device)
        advs      = torch.FloatTensor(advantages).to(device)
        rets      = torch.FloatTensor(returns).to(device)
        advs = (advs - advs.mean()) / (advs.std() + 1e-8)  # normalize
        return states, actions, log_probs, advs, rets

    def clear(self):
        self.__init__()

## PPO Agent

After each rollout, run K mini-batch epochs over the collected data. Shuffling each epoch ensures data diversity. This multi-epoch reuse of on-policy data is what makes PPO more sample-efficient than vanilla policy gradient.

In [ ]:
class PPOAgent:
    def __init__(self, state_dim, action_dim,
                 lr=3e-4, gamma=0.99, lam=0.95,
                 clip_eps=0.2, n_epochs=10, batch_size=64,
                 vf_coef=0.5, ent_coef=0.01):

        self.gamma      = gamma
        self.lam        = lam
        self.clip_eps   = clip_eps
        self.n_epochs   = n_epochs
        self.batch_size = batch_size
        self.vf_coef    = vf_coef
        self.ent_coef   = ent_coef

        self.policy    = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr, eps=1e-5)
        self.buffer    = RolloutBuffer()

    @torch.no_grad()
    def select_action(self, state):
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob, value = self.policy.get_action(s)
        return action.item(), log_prob.item(), value.item()

    def update(self, last_value):
        advantages, returns = self.buffer.compute_gae(last_value, self.gamma, self.lam)
        states, actions, old_log_probs, advs, rets = self.buffer.get_tensors(advantages, returns)

        n = len(states)
        indices = np.arange(n)

        for _ in range(self.n_epochs):
            np.random.shuffle(indices)
            for start in range(0, n, self.batch_size):
                idx = indices[start:start + self.batch_size]

                new_log_probs, values, entropy = self.policy.evaluate(states[idx], actions[idx])

                # Probability ratio
                ratio = torch.exp(new_log_probs - old_log_probs[idx])

                # Clipped surrogate loss
                surr1 = ratio * advs[idx]
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advs[idx]
                policy_loss  = -torch.min(surr1, surr2).mean()

                # Value loss
                value_loss   = F.mse_loss(values, rets[idx])

                # Entropy bonus (encourages exploration)
                entropy_loss = -entropy.mean()

                loss = policy_loss + self.vf_coef * value_loss + self.ent_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optimizer.step()

        self.buffer.clear()

## Training

PPO collects `ROLLOUT_STEPS` timesteps (may span multiple episodes), then updates once, then collects again. Unlike DQN, old data is thrown away after each update.

In [ ]:
env = gym.make('CartPole-v1')
state_dim  = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = PPOAgent(state_dim, action_dim)

ROLLOUT_STEPS = 512
TOTAL_STEPS   = 200_000

episode_rewards   = []
current_ep_reward = 0
state, _ = env.reset(seed=42)
global_step = 0

while global_step < TOTAL_STEPS:
    for _ in range(ROLLOUT_STEPS):
        action, log_prob, value = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.buffer.push(state, action, log_prob, reward, value, float(done))
        state = next_state
        current_ep_reward += reward
        global_step += 1

        if done:
            episode_rewards.append(current_ep_reward)
            current_ep_reward = 0
            state, _ = env.reset()

    # Bootstrap value of last observed state
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        _, last_value = agent.policy(s)
        last_value = last_value.item()

    agent.update(last_value)

    if len(episode_rewards) >= 10:
        avg = np.mean(episode_rewards[-10:])
        print(f'Step {global_step:7d}/{TOTAL_STEPS}  Eps: {len(episode_rewards):4d}  Avg(10): {avg:6.1f}')

env.close()

## Results

In [ ]:
def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, color='seagreen', label='Reward')
if len(episode_rewards) >= 20:
    plt.plot(np.arange(len(smooth(episode_rewards)))+19, smooth(episode_rewards),
             color='seagreen', linewidth=2, label='Smoothed (20-ep)')
plt.axhline(195, color='red', linestyle='--', label='Solved (195)')
plt.xlabel('Episode'); plt.ylabel('Total Reward')
plt.title('PPO on CartPole-v1')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Best avg (last 50): {np.mean(episode_rewards[-50:]):.1f}')

## Evaluate

In [ ]:
agent.policy.eval()
eval_env = gym.make('CartPole-v1')
eval_rewards = []
for _ in range(20):
    s, _ = eval_env.reset()
    total = 0
    while True:
        with torch.no_grad():
            logits, _ = agent.policy(torch.FloatTensor(s).unsqueeze(0).to(device))
            a = logits.argmax(1).item()  # greedy
        s, r, term, trunc, _ = eval_env.step(a)
        total += r
        if term or trunc: break
    eval_rewards.append(total)
eval_env.close()
print(f'Greedy eval (20 ep): mean={np.mean(eval_rewards):.1f}  std={np.std(eval_rewards):.1f}')

## DQN vs DDQN vs PPO Comparison

| | DQN | DDQN | PPO |
|---|---|---|---|
| **Type** | Value-based | Value-based | Policy-based (Actor-Critic) |
| **Policy** | Implicit (argmax Q) | Implicit (argmax Q) | Explicit pi(a|s) |
| **Data reuse** | Replay buffer (off-policy) | Replay buffer (off-policy) | On-policy rollouts only |
| **Stability trick** | Target network + replay | Decouple select/eval | Clipped ratio |
| **Action spaces** | Discrete only | Discrete only | Discrete and Continuous |
| **Overestimation** | High | Lower | Not applicable |
| **Hyperparams** | Few | Few | More (clip, lambda, epochs, coefs) |

**Use PPO when**: continuous action spaces (MuJoCo, robotics), when you want a stable off-the-shelf algorithm, multi-agent settings.  
**Use DQN/DDQN when**: discrete action games (Atari), simpler environments where a value function suffices.